# Take the frame based detections and turn them into movement trajectories
(For speed, uses multiprocessing and breaks up observations into 10 overlapping temporal groups to process seperately since tracks are much shorter in time than the total length of the obervations.)

In [1]:
import glob
import os

import matplotlib.pyplot as plt
from multiprocessing import Pool
import numpy as np

import bat_functions as kbf

In [18]:
base_folder = r"D:\kasanka-bats"
# Which day to track
day = "30-Oct-2020"
#"20211201"
#"20211207"
#"20211215"
camera_folders = sorted(glob.glob(os.path.join(base_folder, day, '*')))

In [19]:
base_folder = r"D:\kasanka-bats"
# Which days to track
days = ["30-Oct-2020"]
camera_folders = []

for day in days:
    print(day)
    day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, '*')))
    camera_folders.extend(day_camera_folders)
    #day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, 'MusolaTower*')))
    #camera_folders.extend(day_camera_folders)
    #day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, 'MusolaPath2*')))
    #camera_folders.extend(day_camera_folders)
    #day_camera_folders = sorted(glob.glob(os.path.join(base_folder, day, 'Sunset*')))
    #camera_folders.extend(day_camera_folders)

30-Oct-2020


In [20]:
print(camera_folders)

['D:\\kasanka-bats\\30-Oct-2020\\BBC', 'D:\\kasanka-bats\\30-Oct-2020\\Chinyangali', 'D:\\kasanka-bats\\30-Oct-2020\\Fibwe Parking', 'D:\\kasanka-bats\\30-Oct-2020\\Fibwe Public', 'D:\\kasanka-bats\\30-Oct-2020\\Musola Parking', 'D:\\kasanka-bats\\30-Oct-2020\\Musola Path', 'D:\\kasanka-bats\\30-Oct-2020\\Musola Tower', 'D:\\kasanka-bats\\30-Oct-2020\\Not Chingyangali', 'D:\\kasanka-bats\\30-Oct-2020\\Puku', 'D:\\kasanka-bats\\30-Oct-2020\\Sunset']


In [21]:
n_camera_folders = []
for folder in camera_folders:
    tracks_file = os.path.join(folder, "raw_tracks.npy")
    if os.path.exists(tracks_file):
        # Skip videos that have already been tracked
        continue
    else:
        n_camera_folders.append(folder)

print("Videos to track...")
print(*n_camera_folders, sep='\n')

Videos to track...
D:\kasanka-bats\30-Oct-2020\BBC
D:\kasanka-bats\30-Oct-2020\Chinyangali
D:\kasanka-bats\30-Oct-2020\Fibwe Parking
D:\kasanka-bats\30-Oct-2020\Fibwe Public
D:\kasanka-bats\30-Oct-2020\Musola Parking
D:\kasanka-bats\30-Oct-2020\Musola Path
D:\kasanka-bats\30-Oct-2020\Musola Tower
D:\kasanka-bats\30-Oct-2020\Not Chingyangali
D:\kasanka-bats\30-Oct-2020\Puku
D:\kasanka-bats\30-Oct-2020\Sunset


In [22]:
def track(camera_dict):
    camera_folder = camera_dict['camera_folder']
    if glob.glob(os.path.join(camera_folder, 'first_frame*.npy')):
        print(f"Skipping {os.path.basename(camera_folder)} since files exist.")
        return
    first_frame = camera_dict['first_frame']
    max_frame = camera_dict['max_frame']
    print(f"{os.path.basename(camera_folder)} begun.")
    contours_files = sorted(
        glob.glob(os.path.join(camera_folder, 'contours-compressed-*.npy'))
    )
    if contours_files:
        contours_files = contours_files[1:]
        centers = np.load(os.path.join(camera_folder, 'centers.npy'), allow_pickle=True)
        sizes = np.load(os.path.join(camera_folder, 'size.npy'), allow_pickle=True)
        tracks_file = os.path.join(camera_folder, f'first_frame_{first_frame}_max_val_{max_frame}_raw_tracks.npy')
        raw_tracks = kbf.find_tracks(first_frame, centers, contours_files=contours_files, 
                                     sizes_list=sizes, tracks_file=tracks_file,
                                     max_frame=max_frame)
    else:
        print("Missing contour files.")

In [23]:
camera_dicts = []
for camera_folder in n_camera_folders:
    # To speed up processing, detections found in each observation are split
    # into 10 groups by time with 15 seconds of overlap in each group
    centers_file = os.path.join(camera_folder, 'centers.npy')
    centers = np.load(centers_file, allow_pickle=True)
    max_vals = np.linspace(0, len(centers), 10, dtype=int)[1:].tolist()
    max_vals[-1] = None
    min_vals = np.linspace(0, len(centers), 10, dtype=int)[:-1]
    # 15 second overlap
    min_vals[1:] = min_vals[1:] - 450
    for min_val, max_val in zip(min_vals, max_vals):
        min_val = np.max([min_val, 0])
        camera_dict = {'camera_folder': camera_folder,
                       'first_frame': min_val,
                       'max_frame': max_val}
        if max_val is None:
            tracks_basename = f'first_frame_{min_val:06d}_max_val_{max_val}_raw_tracks.npy'
        else:
            tracks_basename = f'first_frame_{min_val:06d}_max_val_{max_val:06d}_raw_tracks.npy'
        tracks_file = os.path.join(camera_folder, tracks_basename)
        if not os.path.exists(tracks_file):
            print(tracks_file)
            camera_dicts.append(camera_dict)
        

D:\kasanka-bats\30-Oct-2020\BBC\first_frame_000000_max_val_005653_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_005203_max_val_011307_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_010857_max_val_016961_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_016511_max_val_022615_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_022165_max_val_028269_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_027819_max_val_033923_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_033473_max_val_039577_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_039127_max_val_045231_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\BBC\first_frame_044781_max_val_None_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\Chinyangali\first_frame_000000_max_val_004830_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\Chinyangali\first_frame_004380_max_val_009660_raw_tracks.npy
D:\kasanka-bats\30-Oct-2020\Chinyangali\first_frame_009210_max_val_014490_raw_tracks.npy
D:\kas

In [24]:
print(camera_dicts)
#with Pool(processes=5) as pool:
#    pool.map(track, camera_dicts)

[{'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 0, 'max_frame': 5653}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 5203, 'max_frame': 11307}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 10857, 'max_frame': 16961}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 16511, 'max_frame': 22615}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 22165, 'max_frame': 28269}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 27819, 'max_frame': 33923}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 33473, 'max_frame': 39577}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 39127, 'max_frame': 45231}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\BBC', 'first_frame': 44781, 'max_frame': None}, {'camera_folder': 'D:\\kasanka-bats\\30-Oct-2020\\Chinyangali', 'first_frame': 0, 'max_frame': 4830}, {'camera_fold

In [25]:

for camera_dict in camera_dicts:
    track(camera_dict)

BBC begun.
using D:\kasanka-bats\30-Oct-2020\BBC\contours-compressed-01.npy
frame 0 processed.
BBC final save.
Skipping BBC since files exist.
Skipping BBC since files exist.
Skipping BBC since files exist.
Skipping BBC since files exist.
Skipping BBC since files exist.
Skipping BBC since files exist.
Skipping BBC since files exist.
Skipping BBC since files exist.
Chinyangali begun.
using D:\kasanka-bats\30-Oct-2020\Chinyangali\contours-compressed-01.npy
frame 0 processed.
Chinyangali final save.
Skipping Chinyangali since files exist.
Skipping Chinyangali since files exist.
Skipping Chinyangali since files exist.
Skipping Chinyangali since files exist.
Skipping Chinyangali since files exist.
Skipping Chinyangali since files exist.
Skipping Chinyangali since files exist.
Skipping Chinyangali since files exist.
Fibwe Parking begun.
using D:\kasanka-bats\30-Oct-2020\Fibwe Parking\contours-compressed-01.npy
frame 0 processed.
Fibwe Parking final save.
Skipping Fibwe Parking since files ex

### Now connect the 10 sections of the observation that were tracked seperately together

In [26]:
def combine_overlapping_tracks(observation_folder, first_group=0, last_group=None, save=False):
    track_files = glob.glob(os.path.join(observation_folder, 'first_frame*.npy'))
    track_files = sorted(track_files, key=lambda f: int(f.split('_')[-6]))

    track_groups = []
    for file in track_files:
        track_groups.append(np.load(file, allow_pickle=True))
        
        
    for track_file in track_files:
        print(os.path.basename(track_file))
        
    first_overlap_frames = [int(f.split('_')[-6]) for f in track_files[1:]]
    first_overlap_frames.append(None)
    print(first_overlap_frames)
        
    all_tracks = []

    total_tracks = 0
    for group_ind, track_group in enumerate(track_groups[first_group:last_group]):
        if group_ind >= len(track_groups) -1:
            for track in track_group:
                if type(track['track']) == list:
                    track['track'] = np.stack(track['track'])
                    track['pos_index'] = np.stack(track['pos_index'])
                    if 'size' in track:
                        track['size'] = np.stack(track['size'])
                all_tracks.append(track)
            total_tracks += len(track_group)
            break

        for track_ind, track in enumerate(track_group):
            if track['first_frame'] < first_overlap_frames[first_group + group_ind]:
                all_tracks.append(track)


    all_tracks_file = os.path.join(observation_folder, 'raw_tracks.npy')
    if save:
        np.save(all_tracks_file, all_tracks)
        print('saved')

In [27]:
observation_folders = []
for folder in camera_folders:
    if not os.path.exists(os.path.join(folder, 'first_frame*.npy')):
        observation_folders.append(folder)

In [28]:
print(observation_folders)

['D:\\kasanka-bats\\30-Oct-2020\\BBC', 'D:\\kasanka-bats\\30-Oct-2020\\Chinyangali', 'D:\\kasanka-bats\\30-Oct-2020\\Fibwe Parking', 'D:\\kasanka-bats\\30-Oct-2020\\Fibwe Public', 'D:\\kasanka-bats\\30-Oct-2020\\Musola Parking', 'D:\\kasanka-bats\\30-Oct-2020\\Musola Path', 'D:\\kasanka-bats\\30-Oct-2020\\Musola Tower', 'D:\\kasanka-bats\\30-Oct-2020\\Not Chingyangali', 'D:\\kasanka-bats\\30-Oct-2020\\Puku', 'D:\\kasanka-bats\\30-Oct-2020\\Sunset']


In [29]:
for folder in observation_folders:
    combine_overlapping_tracks(folder, save=True)

first_frame_0_max_val_5653_raw_tracks.npy
[None]
saved
first_frame_0_max_val_4830_raw_tracks.npy
[None]
saved
first_frame_0_max_val_4513_raw_tracks.npy
[None]
saved
first_frame_0_max_val_4057_raw_tracks.npy
[None]
saved
first_frame_0_max_val_3887_raw_tracks.npy
[None]
saved
first_frame_0_max_val_4016_raw_tracks.npy
[None]
saved
first_frame_0_max_val_4726_raw_tracks.npy
[None]
saved
first_frame_0_max_val_2356_raw_tracks.npy
[None]
saved
first_frame_0_max_val_3206_raw_tracks.npy
[None]
saved
first_frame_0_max_val_3206_raw_tracks.npy
[None]
saved
